# Bước 2 — Sentiment Analysis trên review

Chấm sentiment cho các review có bình luận text, dùng làm cơ sở tính
`adjusted_score` ở bước 3 (`39_als_train.ipynb`).

**Thay đổi so với bản cũ:**

1. **Bỏ hẳn bước dịch PT→EN bằng `deep-translator`.** Cách cũ ghép 20 câu
   bằng delimiter `" ||| "` rồi dịch rồi tách lại — rủi ro lệch dòng nếu
   Google Translate không giữ nguyên delimiter, làm sai lệch toàn bộ dữ liệu
   sentiment phía sau. Thay vào đó dùng thẳng **model sentiment đa ngôn ngữ**
   chạy trực tiếp trên tiếng Bồ Đào Nha gốc — không cần dịch, không có bước
   trung gian nào có thể làm lệch dữ liệu.
2. **Model đa lớp (1–5 sao) thay vì nhị phân POSITIVE/NEGATIVE.** Model cũ
   (`distilbert-sst2`) chỉ có 2 lớp và train trên review phim ảnh (miền dữ
   liệu khác hẳn review thương mại điện tử). Model mới:
   `nlptown/bert-base-multilingual-uncased-sentiment` — đa ngôn ngữ (có tiếng
   Bồ), output trực tiếp thang 1–5 sao, khớp thang `review_score` luôn.
3. **Không tính sẵn `adjusted_score` với `ALPHA` cố định.** Notebook này chỉ
   xuất `sentiment_score` (0–1, chuẩn hoá từ số sao). Việc chọn `ALPHA` tối ưu
   được làm bằng grid-search ở `39_als_train.ipynb` (đánh giá qua
   Precision@10/NDCG@10 thay vì đoán số) — xem README mục "Hạn chế đã biết"
   trước đây.
4. **`is_fake` vẫn là heuristic** (`|sentiment*5 - review_score| > 2`), nhưng
   thêm 1 cell lấy mẫu ngẫu nhiên để bạn tự kiểm tra thủ công độ chính xác của
   ngưỡng này, thay vì tin tưởng mù quáng.
5. Ghi MongoDB theo kiểu idempotent (giống notebook 1), tránh mất dữ liệu nếu
   lỗi giữa chừng.


In [2]:
import sys
sys.path.append(".")
from config import PROCESSED_DIR, MONGO_URI, MONGO_DB, get_processed_path

print("PROCESSED_DIR =", PROCESSED_DIR)
print("MONGO_URI     =", MONGO_URI)


PROCESSED_DIR = C:\Users\MY LAPTOP\PycharmProjects\Sentiment\data\processed
MONGO_URI     = mongodb://localhost:27017/


## Lấy review có nội dung bình luận từ MongoDB

In [3]:
from pymongo import MongoClient
import pandas as pd

client = MongoClient(MONGO_URI)
db = client[MONGO_DB]

cursor = db["reviews"].find(
    {"review_comment_message": {"$exists": True, "$ne": None, "$nin": ["", " "]}},
    {"_id": 0, "review_id": 1, "order_id": 1, "review_score": 1, "review_comment_message": 1},
)
reviews_pd = pd.DataFrame(list(cursor))
client.close()

# QUAN TRỌNG: MongoDB/BSON có thể trả review_score về dưới dạng string tuỳ
# theo kiểu Spark đã ghi ở bước load_to_mongo (int32/int64/string đều có thể
# gặp). pandas 3.x (backend PyArrow) KHÔNG tự ép kiểu ngầm như bản cũ, nên
# phép trừ int - str sẽ lỗi TypeError ở bước tính is_fake bên dưới nếu không
# ép kiểu tường minh ngay từ đây.
reviews_pd["review_score"] = pd.to_numeric(reviews_pd["review_score"], errors="coerce")
reviews_pd["review_id"] = reviews_pd["review_id"].astype(str)
reviews_pd["order_id"] = reviews_pd["order_id"].astype(str)
reviews_pd["review_comment_message"] = reviews_pd["review_comment_message"].astype(str)

n_before = len(reviews_pd)
reviews_pd = reviews_pd.dropna(subset=["review_comment_message", "review_score"])
reviews_pd = reviews_pd[reviews_pd["review_comment_message"].str.strip() != ""]
reviews_pd = reviews_pd.reset_index(drop=True)
n_after = len(reviews_pd)
if n_after < n_before:
    print(f"CẢNH BÁO: đã bỏ {n_before - n_after:,} dòng do review_score không phải số hợp lệ hoặc bình luận rỗng")

reviews_pd["review_score"] = reviews_pd["review_score"].astype(int)

print(f"Số review có bình luận: {len(reviews_pd):,}")
print(reviews_pd.dtypes)
reviews_pd.head()


CẢNH BÁO: đã bỏ 106 dòng do review_score không phải số hợp lệ hoặc bình luận rỗng
Số review có bình luận: 40,967
review_id                   str
order_id                    str
review_score              int64
review_comment_message      str
dtype: object


,review_id,order_id,review_score,review_comment_message
0,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,Recebi bem antes do prazo estipulado.
1,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,Parabéns lojas lannister adorei comprar pela I...
2,8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,aparelho eficiente. no site a marca do aparelh...
3,4b49719c8a200003f700d3d986ea1a19,9d6f15f95d01e79bd1349cc208361f09,4,"Mas um pouco ,travando...pelo valor ta Boa."
4,3948b09f7c818e2d86c9a546758b2335,e51478e7e277a83743b6f9991dbfa3fb,5,"Vendedor confiável, produto ok e entrega antes..."


## Chấm sentiment (đa ngôn ngữ, đa lớp — không dịch)

`nlptown/bert-base-multilingual-uncased-sentiment` trả về label dạng
`"1 star"` .. `"5 stars"`. Lấy số sao trực tiếp, KHÔNG cần bước dịch trung
gian nào.

Batch theo GPU/CPU batch size của `transformers` (an toàn — không tự ghép
chuỗi bằng delimiter như bản cũ), và có retry cho từng câu lẻ nếu 1 dòng
trong batch lỗi (thay vì mất nguyên batch 20 dòng như trước).

In [4]:
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
print("Device:", "GPU" if device == 0 else "CPU")

sentiment_model = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    tokenizer="nlptown/bert-base-multilingual-uncased-sentiment",
    truncation=True,
    max_length=512,
    device=device,
)

def label_to_stars(label: str) -> int:
    # label dạng "3 stars" hoặc "1 star"
    return int(label.split()[0])

def score_sentiment_batch(texts: list[str], batch_size: int = 16) -> list[int | None]:
    """Chấm sentiment cho 1 list câu. Nếu cả batch lỗi, fallback chấm
    từng câu lẻ (retry) thay vì trả None cho toàn bộ batch."""
    try:
        results = sentiment_model(texts, batch_size=batch_size, truncation=True)
        return [label_to_stars(r["label"]) for r in results]
    except Exception as e:
        print(f"[CẢNH BÁO] Batch {len(texts)} câu lỗi ({e}), chuyển sang chấm từng câu lẻ...")
        stars = []
        for t in texts:
            try:
                r = sentiment_model(t, truncation=True)[0]
                stars.append(label_to_stars(r["label"]))
            except Exception as e2:
                print(f"  -> bỏ qua 1 câu lỗi: {e2}")
                stars.append(None)
        return stars


Device: CPU


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [5]:
BATCH_SIZE = 16
texts = reviews_pd["review_comment_message"].astype(str).str.slice(0, 512).tolist()

all_stars: list[int | None] = []
for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i : i + BATCH_SIZE]
    all_stars.extend(score_sentiment_batch(batch, batch_size=BATCH_SIZE))
    if (i // BATCH_SIZE) % 20 == 0:
        print(f"Đã xử lý {i + len(batch):,}/{len(texts):,} review")

reviews_pd["sentiment_stars"] = all_stars
n_failed = reviews_pd["sentiment_stars"].isna().sum()
print(f"Số review chấm lỗi (bị bỏ qua): {n_failed:,} / {len(reviews_pd):,}")

reviews_pd = reviews_pd.dropna(subset=["sentiment_stars"]).copy()
reviews_pd["sentiment_stars"] = reviews_pd["sentiment_stars"].astype(int)

# Chuẩn hoá 1-5 sao -> 0-1, để tương thích với công thức adjusted_score cũ
# (adjusted_score sẽ được tính bên notebook als_train, xem markdown ở trên)
reviews_pd["sentiment_score"] = (reviews_pd["sentiment_stars"] - 1) / 4.0

reviews_pd[["review_score", "sentiment_stars", "sentiment_score"]].describe()


Đã xử lý 16/40,967 review
Đã xử lý 336/40,967 review
Đã xử lý 656/40,967 review
Đã xử lý 976/40,967 review
Đã xử lý 1,296/40,967 review
Đã xử lý 1,616/40,967 review
Đã xử lý 1,936/40,967 review
Đã xử lý 2,256/40,967 review
Đã xử lý 2,576/40,967 review
Đã xử lý 2,896/40,967 review
Đã xử lý 3,216/40,967 review
Đã xử lý 3,536/40,967 review
Đã xử lý 3,856/40,967 review
Đã xử lý 4,176/40,967 review
Đã xử lý 4,496/40,967 review
Đã xử lý 4,816/40,967 review
Đã xử lý 5,136/40,967 review
Đã xử lý 5,456/40,967 review
Đã xử lý 5,776/40,967 review
Đã xử lý 6,096/40,967 review
Đã xử lý 6,416/40,967 review
Đã xử lý 6,736/40,967 review
Đã xử lý 7,056/40,967 review
Đã xử lý 7,376/40,967 review
Đã xử lý 7,696/40,967 review
Đã xử lý 8,016/40,967 review
Đã xử lý 8,336/40,967 review
Đã xử lý 8,656/40,967 review
Đã xử lý 8,976/40,967 review
Đã xử lý 9,296/40,967 review
Đã xử lý 9,616/40,967 review
Đã xử lý 9,936/40,967 review
Đã xử lý 10,256/40,967 review
Đã xử lý 10,576/40,967 review
Đã xử lý 10,896/40,96

,review_score,sentiment_stars,sentiment_score
count,40967.000000,40967.000000,40967.000000
mean,3.669612,3.262040,0.565510
std,1.615519,1.742449,0.435612
min,1.000000,1.000000,0.000000
25%,2.000000,1.000000,0.000000
50%,5.000000,4.000000,0.750000
75%,5.000000,5.000000,1.000000
max,5.000000,5.000000,1.000000


## `is_fake` — heuristic, kèm bước tự kiểm tra thủ công

Ngưỡng `> 2` (trên thang 5 sao) vẫn là heuristic, **chưa có nhãn thật để
validate**. Thêm cell lấy mẫu ngẫu nhiên các review bị gắn cờ để bạn đọc và tự
đánh giá xem ngưỡng này có hợp lý không, trước khi dùng cột `is_fake` cho bất
kỳ kết luận nào trong báo cáo.

In [6]:
FAKE_THRESHOLD = 2.0  # thang 5 sao — heuristic, xem ghi chú ở trên

reviews_pd["is_fake"] = (
    (reviews_pd["sentiment_stars"] - reviews_pd["review_score"]).abs() > FAKE_THRESHOLD
)

print(f"Tỉ lệ review bị gắn cờ is_fake: {reviews_pd['is_fake'].mean():.2%}")


Tỉ lệ review bị gắn cờ is_fake: 9.72%


In [7]:
# Lấy mẫu ngẫu nhiên để TỰ KIỂM TRA THỦ CÔNG — không tin tưởng mù quáng vào ngưỡng heuristic
sample_flagged = reviews_pd[reviews_pd["is_fake"]].sample(
    n=min(15, reviews_pd["is_fake"].sum()), random_state=42
)[["review_score", "sentiment_stars", "review_comment_message"]]

with pd.option_context("display.max_colwidth", 120):
    display(sample_flagged)

print(
    "\nĐọc thủ công các dòng trên: nếu phần lớn thực sự trông 'giả'/lệch cảm xúc "
    "bất thường so với điểm sao, ngưỡng FAKE_THRESHOLD=2.0 tạm chấp nhận được. "
    "Nếu nhiều dòng trông bình thường (ví dụ khách chê 1 chi tiết nhỏ nhưng vẫn "
    "cho 5 sao), nên tăng ngưỡng hoặc bỏ hẳn is_fake khỏi báo cáo cho tới khi có "
    "tập nhãn thật để validate (xem paper Ott et al., 2011 — ACL, về phát hiện "
    "deceptive review, nếu muốn làm bài bản hơn)."
)


,review_score,sentiment_stars,review_comment_message
4120,5,1,Entrega antes do prazo!!! Recebi o produto em 03 dias!!!
25738,5,2,Ótimo
6915,5,1,Não recomendaria simplesmente pelo fato dos envios poderem ter o mesmo problema que o meu.
32061,5,1,"ótima compra, chegou antes do prazo"
23902,5,1,o produto chegou com quase 15 dias de antecedência.
17671,5,2,Só achei muito pequeno e a grama MUITO ruim
37683,4,1,"Recebi incompleto, faltou um produto"
31214,4,1,Chegou antes até do prazo
4024,4,1,o problema foi com os Correios que continua com um serviço vergonhoso.
16344,1,5,"estou muito satisfeita, parabens a toda equipe"



Đọc thủ công các dòng trên: nếu phần lớn thực sự trông 'giả'/lệch cảm xúc bất thường so với điểm sao, ngưỡng FAKE_THRESHOLD=2.0 tạm chấp nhận được. Nếu nhiều dòng trông bình thường (ví dụ khách chê 1 chi tiết nhỏ nhưng vẫn cho 5 sao), nên tăng ngưỡng hoặc bỏ hẳn is_fake khỏi báo cáo cho tới khi có tập nhãn thật để validate (xem paper Ott et al., 2011 — ACL, về phát hiện deceptive review, nếu muốn làm bài bản hơn).


## Lưu kết quả: parquet + MongoDB (idempotent)

In [8]:
OUT_COLUMNS = [
    "review_id", "order_id", "review_score",
    "sentiment_stars", "sentiment_score", "is_fake",
]
out_df = reviews_pd[OUT_COLUMNS].reset_index(drop=True)

parquet_path = get_processed_path("reviews_sentiment.parquet")
out_df.to_parquet(parquet_path, index=False)
print(f"Đã lưu {len(out_df):,} dòng vào {parquet_path}")


Đã lưu 40,967 dòng vào C:\Users\MY LAPTOP\PycharmProjects\Sentiment\data\processed\reviews_sentiment.parquet


In [9]:
from pymongo.errors import PyMongoError

def write_pandas_idempotent(df: pd.DataFrame, mongo_uri: str, db_name: str, collection: str):
    tmp_name = f"{collection}_tmp"
    client = MongoClient(mongo_uri)
    try:
        db = client[db_name]
        db[tmp_name].drop()
        records = df.to_dict("records")
        if records:
            db[tmp_name].insert_many(records)
        db[tmp_name].rename(collection, dropTarget=True)
        print(f"OK: {collection} đã được cập nhật ({db[collection].estimated_document_count():,} documents)")
    except PyMongoError as e:
        print(f"[LỖI] Ghi '{collection}' thất bại, dữ liệu cũ KHÔNG bị thay đổi: {e}")
        raise
    finally:
        client.close()

write_pandas_idempotent(out_df, MONGO_URI, MONGO_DB, "reviews_sentiment")


OK: reviews_sentiment đã được cập nhật (40,967 documents)
